# 02. Automated Data Quality & Integrity Audit

## Objectives
1. Perform null check, duplicate check, and negative value verification.
2. Assert referential integrity across all foreign keys.
3. Validate temporal logic (`actual_delivery_date >= po_date`).
4. Audit mathematical conservation of daily stock balances: `closing = opening + received - sold - damaged`.

In [ ]:
import pandas as pd
import numpy as np
import os
import sys

sys.path.append(os.path.abspath('..'))
from src.etl.load_data import load_raw_csvs
from src.etl.clean_data import clean_all_data
from src.etl.validate_data import validate_all_data

raw_data = load_raw_csvs()
cleaned_data = clean_all_data(raw_data)
summary_df, is_valid = validate_all_data(cleaned_data)

print(f"\nAll Automated Quality Checks Passed: {is_valid}")
summary_df

### Audit Inventory Conservation Equation Discrepancies

In [ ]:
df_inv = cleaned_data['inventory'].copy()
df_inv['calculated_closing'] = df_inv['opening_stock'] + df_inv['stock_received'] - df_inv['quantity_sold'] - df_inv['damaged_stock']
df_inv['discrepancy'] = df_inv['closing_stock'] - np.maximum(0, df_inv['calculated_closing'])

print(f"Total Discrepancy Count: {(df_inv['discrepancy'] != 0).sum()}")
print(f"Max Absolute Discrepancy: {df_inv['discrepancy'].abs().max()}")